<a href="https://colab.research.google.com/github/jegankanshika/AI/blob/main/LLM_dev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM-dev: Building a GPT Language Model from Scratch

This notebook walks through building a character-level GPT model trained on Shakespeare's text.

---
**Sections:**
1. Download & Explore the Dataset
2. Tokenization
3. Train/Validation Split
4. Data Batching
5. Bigram Language Model (Baseline)
6. The Mathematical Trick Behind Attention
7. Self-Attention Head
8. Full GPT Model
9. Training & Text Generation

## Section 1: Download & Explore the Dataset

We download the **Tiny Shakespeare** dataset — ~1MB of Shakespeare plays used as training text.

In [ ]:
# Download the Tiny Shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [ ]:
# Read and inspect the dataset
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print('Length of dataset in characters:', len(text))
print()
print('--- First 1000 characters ---')
print(text[:1000])

## Section 2: Tokenization

We use **character-level tokenization** — every unique character gets a unique integer ID.
The model only works with numbers, so we need to convert text <-> integers.

In [ ]:
# Build the vocabulary: all unique characters in the dataset
chars = sorted(list(set(text)))
vocab_size = len(chars)

print('All characters in vocabulary:')
print(''.join(chars))
print()
print('Vocabulary size:', vocab_size)

In [ ]:
# Create character <-> integer mappings
stoi = { ch: i for i, ch in enumerate(chars) }  # char -> int
itos = { i: ch for i, ch in enumerate(chars) }  # int -> char

# Encoder: string to list of integers
encode = lambda s: [stoi[c] for c in s]

# Decoder: list of integers to string
decode = lambda l: ''.join([itos[i] for i in l])

# Test encode/decode
print('Encoded "hello world":', encode('hello world'))
print('Decoded back:         ', decode(encode('hello world')))

## Section 3: Train / Validation Split

We encode the full dataset and split into:
- **90% Training** — used to teach the model
- **10% Validation** — used to check how well it learned on unseen data

In [ ]:
import torch

# Encode entire dataset into a PyTorch tensor
data = torch.tensor(encode(text), dtype=torch.long)
print('Data shape:', data.shape, '| Data type:', data.dtype)
print('First 100 encoded tokens:', data[:100])

# 90% train, 10% validation split
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

print(f'\nTraining tokens:   {len(train_data):,}')
print(f'Validation tokens: {len(val_data):,}')

## Section 4: Data Batching

The model processes small **batches** of short **chunks** (sequences).

- `block_size` = how many characters the model sees at once (context window)
- `batch_size` = how many sequences are processed in parallel

In [ ]:
# Demonstrate how a single block creates multiple training examples
block_size = 8

x = train_data[:block_size]        # input
y = train_data[1:block_size + 1]   # target (shifted by 1)

print('Every position creates a training example:')
for t in range(block_size):
    context = x[:t+1].tolist()
    target  = y[t].item()
    print(f'  Input: {context} -> Target: {target} ("{decode([target])}")')

In [ ]:
torch.manual_seed(1337)

batch_size = 4   # sequences processed in parallel
block_size = 8   # context window length

def get_batch(split):
    """Generate a random batch of input/target pairs."""
    d  = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x  = torch.stack([d[i:i+block_size]     for i in ix])
    y  = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('Input  shape:', xb.shape)
print('Target shape:', yb.shape)
print('\nInput batch:\n', xb)
print('\nTarget batch:\n', yb)

## Section 5: Bigram Language Model (Baseline)

The simplest possible model: predicts the next character based **only on the current character**.
No context, no memory — just a lookup table.

In [ ]:
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)  # (B, T, vocab_size)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits  = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits    = logits[:, -1, :]
            probs     = F.softmax(logits, dim=-1)
            idx_next  = torch.multinomial(probs, num_samples=1)
            idx       = torch.cat((idx, idx_next), dim=1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print('Logits shape:', logits.shape)
print('Loss:', loss.item(), '  (expected ~4.17 for random init)')

In [ ]:
# Generate BEFORE training
print('=== Generated text BEFORE training ===')
context = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(context, max_new_tokens=300)[0].tolist()))

In [ ]:
# Train the bigram model
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(retain_graph=False)
    loss.backward()
    optimizer.step()
print('Final training loss:', loss.item())

In [ ]:
# Generate AFTER training
print('=== Generated text AFTER training ===')
context = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(context, max_new_tokens=300)[0].tolist()))

## Section 6: The Mathematical Trick Behind Attention

Key insight: tokens can communicate with past tokens efficiently using matrix multiplication.

- **Version 1** — Naive loop (slow)
- **Version 2** — Matrix multiply trick (fast)
- **Version 3** — Softmax + masking (the actual attention way)

In [ ]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)

# VERSION 1: Naive loop
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b, t] = torch.mean(xprev, 0)
print('Version 1 (loop):'); print(xbow[0])

In [ ]:
# VERSION 2: Matrix multiply trick
wei  = torch.tril(torch.ones(T, T))
wei  = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x
print('Version 2 == Version 1?', torch.allclose(xbow, xbow2))
print('Weight matrix:'); print(wei)

In [ ]:
# VERSION 3: Softmax with masking
tril = torch.tril(torch.ones(T, T))
wei  = torch.zeros((T, T))
wei  = wei.masked_fill(tril == 0, float('-inf'))
wei  = F.softmax(wei, dim=-1)
xbow3 = wei @ x
print('Version 3 == Version 1?', torch.allclose(xbow, xbow3))
print('Attention weights:'); print(wei)

## Section 7: Self-Attention Head

Now we make the weights **data-dependent** using **Query**, **Key**, and **Value** vectors.
Each token learns which past tokens to pay attention to.

In [ ]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

head_size = 16
key   = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)    # (B, T, head_size) -- what do I contain?
q = query(x)  # (B, T, head_size) -- what am I looking for?

# Scaled dot-product attention
wei = q @ k.transpose(-2, -1) * head_size**-0.5  # (B, T, T)
tril = torch.tril(torch.ones(T, T))
wei  = wei.masked_fill(tril == 0, float('-inf'))
wei  = F.softmax(wei, dim=-1)

v   = value(x)
out = wei @ v   # (B, T, head_size)
print('Output shape:', out.shape)
print('Attention weights (first item):'); print(wei[0].detach())

## Section 8: Full GPT Model

Putting it all together:
- Token + Position Embeddings
- Multi-Head Self-Attention
- FeedForward Network
- Residual Connections + Layer Normalization
- Stacked Transformer Blocks

In [ ]:
# ─── Hyperparameters ───────────────────────────────────────────────
batch_size    = 64
block_size    = 256
max_iters     = 5000
eval_interval = 500
learning_rate = 3e-4
device        = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters    = 200
n_embd        = 384
n_head        = 6
n_layer       = 6
dropout       = 0.2
# ───────────────────────────────────────────────────────────────────

torch.manual_seed(1337)
print(f'Device: {device}')
print(f'Model: {n_layer} layers | {n_head} heads | {n_embd} embedding dim')

In [ ]:
# Re-build data splits with new block_size
train_data = data[:int(0.9*len(data))].to(device)
val_data   = data[int(0.9*len(data)):].to(device)

def get_batch(split):
    d  = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x  = torch.stack([d[i:i+block_size]     for i in ix])
    y  = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
class Head(nn.Module):
    """One head of self-attention."""
    def __init__(self, head_size):
        super().__init__()
        self.key     = nn.Linear(n_embd, head_size, bias=False)
        self.query   = nn.Linear(n_embd, head_size, bias=False)
        self.value   = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v   = self.value(x)
        return wei @ v


class MultiHeadAttention(nn.Module):
    """Multiple heads of self-attention in parallel."""
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads   = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj    = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    """Token-wise feedforward network."""
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Transformer block: attention then feedforward, with residuals."""
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa   = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1  = nn.LayerNorm(n_embd)
        self.ln2  = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class GPTLanguageModel(nn.Module):
    """Full GPT Language Model."""
    def __init__(self):
        super().__init__()
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks  = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f    = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits  = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss    = F.cross_entropy(logits, targets)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond  = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits    = logits[:, -1, :]
            probs     = F.softmax(logits, dim=-1)
            idx_next  = torch.multinomial(probs, num_samples=1)
            idx       = torch.cat((idx, idx_next), dim=1)
        return idx


model = GPTLanguageModel().to(device)
num_params = sum(p.numel() for p in model.parameters())
print(f'Model on: {device}')
print(f'Total parameters: {num_params:,}  (~{num_params/1e6:.1f}M)')

## Section 9: Training & Text Generation

Train the full GPT model. Watch both train and val loss decrease.

> **Tip:** Enable GPU in Colab: Runtime → Change runtime type → T4 GPU

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print('Starting training...')
print(f'Training for {max_iters} iterations on {device}')
print('-' * 55)

for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f'Step {iter:5d}/{max_iters} | '
              f'Train loss: {losses["train"]:.4f} | '
              f'Val loss: {losses["val"]:.4f}')
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(retain_graph=False)
    loss.backward()
    optimizer.step()

print('-' * 55)
print('Training complete!')

In [ ]:
# Generate Shakespeare-style text from trained model
print('=== Generated Shakespeare-style text ===')
print()
model.eval()
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated = model.generate(context, max_new_tokens=2000)
print(decode(generated[0].tolist()))

In [ ]:
# (Optional) Save model weights
torch.save(model.state_dict(), 'llm_dev_model.pt')
print('Model saved to llm_dev_model.pt')

# To reload:
# model = GPTLanguageModel().to(device)
# model.load_state_dict(torch.load('llm_dev_model.pt'))
# model.eval()

---

## Summary: What We Built

| Component | Role |
|---|---|
| `token_embedding_table` | Converts character IDs into learned vectors |
| `position_embedding_table` | Adds positional information to each token |
| `Head` | Single self-attention head (Q, K, V) |
| `MultiHeadAttention` | 6 parallel attention heads |
| `FeedForward` | Per-token processing after attention |
| `Block` | Attention + FeedForward + LayerNorm + Residuals |
| `GPTLanguageModel` | Full stack: 6 Blocks + final projection |
| `generate()` | Auto-regressive text generation |

**Key hyperparameters:**
- `n_embd = 384` — embedding dimension
- `n_head = 6` — attention heads  
- `n_layer = 6` — transformer blocks
- `block_size = 256` — context window
- `dropout = 0.2` — regularization

This is the same architecture that powers GPT-2, GPT-3, and modern LLMs — just scaled up!

---
*Based on Andrej Karpathy's [Neural Networks: Zero to Hero](https://github.com/karpathy/nn-zero-to-hero)*